# EDA — Série Histórica de Vazão: Rio Itajaí-Açu em Blumenau

**Estação ANA:** 83500000 — Blumenau  
**Variáveis:** vazão (m³/s) e cota (m)  
**Objetivo:** entender a distribuição, sazonalidade, eventos extremos e gaps
antes de treinar qualquer modelo.

---

### Por que fazer EDA antes de modelar?

Modelos de machine learning são otimizadores de função de perda — eles
não "sabem" que um valor de -999 é um código de missing, ou que uma
vazão de 50.000 m³/s é fisicamente impossível para a bacia. Se você
alimentar dados sujos, o modelo aprende padrões sujos.
A EDA aqui tem três objetivos concretos:
1. Validar que baixamos o que esperávamos (sanity check)
2. Identificar problemas de qualidade (outliers, gaps, mudanças de sensor)
3. Fundamentar as decisões de pré-processamento

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import sys
sys.path.insert(0, '../../')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.ticker as mticker
from matplotlib.patches import FancyArrowPatch
import seaborn as sns
from pathlib import Path

# Estilo consistente em todos os plots
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')
FIGDIR = Path('../../reports/figures')
FIGDIR.mkdir(parents=True, exist_ok=True)

STATION = '83500000'
RAW_DIR = Path('../../data/raw/streamflow')

# Eventos críticos para marcar nos gráficos
FLOOD_EVENTS = {
    '1983-11-09': ('Nov/1983', '17.17 m'),
    '1984-07-07': ('Jul/1984', '16.42 m'),
    '2008-11-23': ('Nov/2008', '11.78 m'),
    '2011-09-08': ('Set/2011', '10.14 m'),
    '2023-09-05': ('Set/2023', '10.55 m'),
}

print('Ambiente configurado.')

## 1. Download dos dados

Se os arquivos já existirem em `data/raw/streamflow/`, esta célula
é um no-op. Caso contrário, faz o download da série completa.

In [ ]:
from src.data.ana_downloader import download_series, save_raw

vazao_path = RAW_DIR / f'{STATION}_vazao_raw.parquet'
cota_path  = RAW_DIR / f'{STATION}_cota_raw.parquet'

if not vazao_path.exists():
    print('Baixando série de vazão...')
    df_vazao = download_series(STATION, 'vazao', start_year=1940)
    save_raw(df_vazao, STATION, 'vazao')
else:
    print(f'Carregando vazão de {vazao_path}')
    df_vazao = pd.read_parquet(vazao_path)

if not cota_path.exists():
    print('Baixando série de cota...')
    df_cota = download_series(STATION, 'cota', start_year=1940)
    save_raw(df_cota, STATION, 'cota')
else:
    print(f'Carregando cota de {cota_path}')
    df_cota = pd.read_parquet(cota_path)

# Alias para variável principal da análise
Q = df_vazao['value'].rename('Q_m3s')
H = df_cota['value'].rename('H_m')

print(f'\nVazão: {Q.index.min().date()} → {Q.index.max().date()} ({len(Q)} dias)')
print(f'Cota:  {H.index.min().date()} → {H.index.max().date()} ({len(H)} dias)')

## 2. Sanity Check — Estatísticas Básicas

Antes de qualquer gráfico: números que nos dizem se o download fez sentido.

In [ ]:
print('=== VAZÃO (m³/s) ===')
print(Q.describe().round(1))
print(f'\nNaN: {Q.isna().sum()} ({Q.isna().mean()*100:.1f}%)')
print(f'Zeros: {(Q == 0).sum()}')
print(f'Negativos: {(Q < 0).sum()}  ← deve ser 0')
print(f'> 15000 m³/s: {(Q > 15000).sum()}  ← fisicamente impossível')

print('\n=== COTA (m) ===')
print(H.describe().round(2))
print(f'\nNaN: {H.isna().sum()} ({H.isna().mean()*100:.1f}%)')
print(f'Máximo histórico esperado: ~17.17 m (nov/1983)')
print(f'Máximo encontrado: {H.max():.2f} m')

## 3. Série Histórica Completa

Gráfico de linha com eventos críticos anotados. Procure por:
- Mudanças abruptas de nível médio (podem indicar troca de sensor)
- Blocos contínuos de NaN (períodos sem medição)
- Picos coerentes com os eventos históricos conhecidos

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(18, 9), sharex=True)

# Vazão
ax = axes[0]
ax.plot(Q.index, Q.values, lw=0.5, color='steelblue', alpha=0.8)
ax.set_ylabel('Vazão (m³/s)', fontsize=11)
ax.set_title('Rio Itajaí-Açu — Estação Blumenau (83500000)', fontsize=13, fontweight='bold')

# Anotar eventos
for date_str, (label, cota_str) in FLOOD_EVENTS.items():
    try:
        date = pd.Timestamp(date_str)
        val = Q.get(date, np.nan)
        if not np.isnan(val):
            ax.annotate(
                f'{label}\nH={cota_str}',
                xy=(date, val), xytext=(0, 30),
                textcoords='offset points',
                arrowprops=dict(arrowstyle='->', color='crimson'),
                fontsize=7, color='crimson', ha='center'
            )
    except Exception:
        pass

# Cota
ax2 = axes[1]
ax2.plot(H.index, H.values, lw=0.5, color='darkorange', alpha=0.8)
ax2.axhline(9.0, color='gold', lw=1, ls='--', label='Alerta (9 m)')
ax2.axhline(11.5, color='red', lw=1, ls='--', label='Emergência (11.5 m)')
ax2.set_ylabel('Cota (m)', fontsize=11)
ax2.set_xlabel('Data', fontsize=11)
ax2.legend(fontsize=9)

fig.tight_layout()
fig.savefig(FIGDIR / '01_serie_historica_completa.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Análise de Completude — Gaps

Mapa de calor anual por mês: ausência de dado em vermelho, presente em azul.

**Por que isso importa para o modelo?**  
O LSTM precisa de sequências contínuas para propagar estado oculto.
Décadas com muitos gaps forçam cortes de sequência, reduzindo o
conjunto efetivo de treinamento. Saber *onde* estão os gaps nos ajuda
a escolher os splits de treino/val/teste.

In [ ]:
# Pivot: linhas = ano, colunas = mês
completeness = (
    Q.resample('MS').apply(lambda s: s.notna().mean())
    .to_frame('completeness')
    .assign(year=lambda d: d.index.year, month=lambda d: d.index.month)
    .pivot(index='year', columns='month', values='completeness')
)
completeness.columns = ['Jan','Fev','Mar','Abr','Mai','Jun',
                        'Jul','Ago','Set','Out','Nov','Dez']

fig, ax = plt.subplots(figsize=(14, max(8, len(completeness) // 4)))
sns.heatmap(
    completeness, ax=ax, cmap='RdYlGn', vmin=0, vmax=1,
    linewidths=0.3, linecolor='white',
    cbar_kws={'label': 'Fração de dias com dado válido'}
)
ax.set_title('Completude da Série de Vazão por Ano/Mês', fontsize=13)
ax.set_ylabel('Ano')
ax.set_xlabel('')
fig.tight_layout()
fig.savefig(FIGDIR / '02_completude_anual.png', dpi=150, bbox_inches='tight')
plt.show()

# Resumo numérico
annual_completeness = completeness.mean(axis=1)
bad_years = annual_completeness[annual_completeness < 0.8]
print(f'Anos com < 80% de completude: {len(bad_years)}')
if not bad_years.empty:
    print(bad_years.round(2).to_string())

## 5. Distribuição e Assimetria

A distribuição de vazão é fundamental para escolher a transformação.

**O que esperamos ver:** distribuição log-normal — fortemente assimétrica
à direita no espaço linear, razoavelmente gaussiana no espaço log.
Se não for assim, precisamos revisar a estratégia de normalização.

In [ ]:
from scipy import stats

Q_clean = Q.dropna()
Q_log = np.log1p(Q_clean)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# 1. Histograma linear
axes[0].hist(Q_clean, bins=100, color='steelblue', edgecolor='none', alpha=0.8)
axes[0].set_xlabel('Vazão (m³/s)')
axes[0].set_ylabel('Frequência')
axes[0].set_title('Espaço Linear')
axes[0].set_yscale('log')
sk = stats.skew(Q_clean)
axes[0].text(0.97, 0.95, f'Skewness = {sk:.2f}', transform=axes[0].transAxes,
             ha='right', va='top', fontsize=9,
             bbox=dict(facecolor='white', alpha=0.7))

# 2. Histograma log
axes[1].hist(Q_log, bins=60, color='seagreen', edgecolor='none', alpha=0.8)
axes[1].set_xlabel('log(1 + Vazão)')
axes[1].set_title('Espaço Logarítmico')
sk_log = stats.skew(Q_log)
axes[1].text(0.97, 0.95, f'Skewness = {sk_log:.2f}', transform=axes[1].transAxes,
             ha='right', va='top', fontsize=9,
             bbox=dict(facecolor='white', alpha=0.7))

# 3. Q-Q plot log
stats.probplot(Q_log, dist='norm', plot=axes[2])
axes[2].set_title('Q-Q Plot (espaço log vs. Normal)')
axes[2].get_lines()[1].set_color('crimson')

fig.suptitle('Distribuição da Vazão — Blumenau', fontsize=13, y=1.02)
fig.tight_layout()
fig.savefig(FIGDIR / '03_distribuicao_vazao.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Skewness linear: {sk:.2f} (ideal para log-transform: > 1)')
print(f'Skewness log:    {sk_log:.2f} (ideal após transform: próximo de 0)')

## 6. Sazonalidade

Boxplot mensal — revela quando as cheias são mais prováveis.

**O que esperamos:**  
Blumenau tem regime subtropical: chuvas orográficas concentradas em
outubro–março. Esperamos boxplots muito mais altos e com caudas
mais longas nesses meses.

**Por que importa para o modelo?**  
O MEF-LSTM não recebe mês como feature explícita, mas recebe atributos
estáticos da bacia. Entender a sazonalidade nos diz se precisamos
incluir alguma encoding temporal (seno/cosseno do dia do ano) como
feature de hindcast.

In [ ]:
df_monthly = Q_clean.to_frame()
df_monthly['month'] = df_monthly.index.month
df_monthly['month_name'] = df_monthly.index.strftime('%b')

month_order = ['Jan','Fev','Mar','Abr','Mai','Jun',
               'Jul','Ago','Set','Out','Nov','Dez']
df_monthly['month_name'] = pd.Categorical(df_monthly['month_name'], categories=month_order)

fig, ax = plt.subplots(figsize=(13, 5))
sns.boxplot(
    data=df_monthly, x='month_name', y='Q_m3s',
    ax=ax, flierprops=dict(marker='.', ms=2, alpha=0.3),
    color='steelblue', linewidth=0.8
)
ax.set_yscale('log')
ax.set_xlabel('')
ax.set_ylabel('Vazão (m³/s) — escala log')
ax.set_title('Sazonalidade da Vazão — Blumenau (escala log)', fontsize=13)

# Destacar meses de alta vazão
for m in [0, 1, 2, 9, 10, 11]:  # Jan, Fev, Mar, Out, Nov, Dez
    ax.axvspan(m - 0.5, m + 0.5, alpha=0.06, color='salmon')

fig.tight_layout()
fig.savefig(FIGDIR / '04_sazonalidade_mensal.png', dpi=150, bbox_inches='tight')
plt.show()

# Médias mensais
print('Mediana por mês (m³/s):')
print(df_monthly.groupby('month')['Q_m3s'].median().round(1).to_string())

## 7. Autocorrelação

Quanto tempo dura a "memória" do rio?

**Por que isso define o hindcast_length no YAML?**  
O LSTM usa uma janela de `hindcast_length` dias como contexto histórico.
Se a autocorrelação cai para zero após 7 dias, não há ganho em usar
30 dias de hindcast — o modelo desperdiçaria parâmetros tentando
aprender de ruído. A janela atual no config é 168h (7 dias diários
ou horários) — vamos verificar se isso é adequado.

In [ ]:
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

Q_log_clean = Q_log.dropna()

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
plot_acf(Q_log_clean, lags=30, ax=axes[0], alpha=0.05)
axes[0].set_title('ACF — Vazão Log (lags em dias)')
axes[0].set_xlabel('Lag (dias)')

plot_pacf(Q_log_clean, lags=30, ax=axes[1], alpha=0.05, method='ywm')
axes[1].set_title('PACF — Vazão Log (lags em dias)')
axes[1].set_xlabel('Lag (dias)')

fig.tight_layout()
fig.savefig(FIGDIR / '05_autocorrelacao.png', dpi=150, bbox_inches='tight')
plt.show()

# Lag onde ACF cai abaixo de 0.5
from statsmodels.tsa.stattools import acf
acf_values = acf(Q_log_clean, nlags=30, fft=True)
lag_half = next((i for i, v in enumerate(acf_values) if v < 0.5), None)
print(f'ACF cai abaixo de 0.5 no lag: {lag_half} dias')
print('→ Sugere hindcast de pelo menos esse número de dias.')

## 8. Análise de Eventos Extremos

Zoom nos cinco maiores eventos históricos. Observamos:
- Taxa de subida (m³/s por dia) — quanto tempo de aviso o modelo precisa dar?
- Forma do hidrograma — pico único vs. múltiplos picos?
- Duração da cheia acima da cota de alerta

In [ ]:
WINDOW_DAYS = 30  # janela em torno do pico

fig, axes = plt.subplots(
    len(FLOOD_EVENTS), 1,
    figsize=(14, 4 * len(FLOOD_EVENTS)),
    sharex=False
)

for ax, (date_str, (label, cota_str)) in zip(axes, FLOOD_EVENTS.items()):
    peak = pd.Timestamp(date_str)
    window = Q.loc[peak - pd.Timedelta(days=WINDOW_DAYS):peak + pd.Timedelta(days=WINDOW_DAYS)]

    if window.empty:
        ax.text(0.5, 0.5, 'Dados indisponíveis', ha='center', va='center',
                transform=ax.transAxes, fontsize=10)
        ax.set_title(f'{label} — sem dados')
        continue

    ax.fill_between(window.index, window.values, alpha=0.3, color='steelblue')
    ax.plot(window.index, window.values, color='steelblue', lw=1.5)
    ax.axvline(peak, color='crimson', lw=1.5, ls='--', label='Pico declarado')

    # Taxa máxima de subida
    diff = window.diff()
    max_rise = diff.max()
    max_rise_date = diff.idxmax()

    ax.set_title(
        f'{label} | cota pico: {cota_str} | Q máx: {window.max():.0f} m³/s '
        f'| subida máx: +{max_rise:.0f} m³/s/dia',
        fontsize=9
    )
    ax.set_ylabel('Q (m³/s)')
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%d/%m'))
    ax.legend(fontsize=8)

    # Preencher período em alerta (proxy: Q > 1000 m³/s)
    ax.axhspan(1000, window.max() * 1.05, alpha=0.04, color='red')

fig.suptitle('Hidrógrafas dos Eventos Extremos — Rio Itajaí-Açu / Blumenau',
             fontsize=13, y=1.01)
fig.tight_layout()
fig.savefig(FIGDIR / '06_eventos_extremos.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Curva Cota × Vazão (Rating Curve)

A relação cota-descarga é a "calibração" da estação: converte nível
(fácil de medir) em vazão (calculado). Problemas nessa curva
aparecem como nuvem de pontos dispersa ou mudanças de regime ao
longo do tempo (remoção de sedimentos, obras na calha, etc.).

In [ ]:
# Alinhar séries pelo índice
df_hq = pd.concat([H, Q], axis=1).dropna()

if len(df_hq) < 10:
    print('Dados insuficientes para curva cota-descarga. Verifique se ambas as séries foram baixadas.')
else:
    fig, ax = plt.subplots(figsize=(9, 6))

    # Colorir por década para detectar mudanças
    df_hq['decade'] = (df_hq.index.year // 10) * 10
    cmap = plt.cm.get_cmap('plasma', df_hq['decade'].nunique())

    for i, (decade, grp) in enumerate(df_hq.groupby('decade')):
        ax.scatter(grp['H_m'], grp['Q_m3s'], s=2, alpha=0.4,
                   color=cmap(i), label=str(decade))

    ax.set_xlabel('Cota (m)')
    ax.set_ylabel('Vazão (m³/s)')
    ax.set_yscale('log')
    ax.set_title('Curva Cota × Descarga por Década', fontsize=12)
    ax.legend(title='Década', fontsize=8, markerscale=4)
    fig.tight_layout()
    fig.savefig(FIGDIR / '07_curva_cota_descarga.png', dpi=150, bbox_inches='tight')
    plt.show()

    r = df_hq[['H_m', 'Q_m3s']].corr().loc['H_m', 'Q_m3s']
    print(f'Correlação H×Q: {r:.3f}')

## 10. Série de Máximos Anuais (Análise de Frequência)

Quanto frequente é uma cheia do porte de 1983?  
Ajustamos uma distribuição GEV (Generalized Extreme Value) — a mais
usada em hidrologia de frequência. O período de retorno estimado
ajuda a contextualizar o desempenho do modelo: errar uma cheia de
100 anos é muito mais custoso que errar um evento de 2 anos.

In [ ]:
from scipy.stats import genextreme

# Máximo anual
Q_annual_max = Q_clean.resample('YE').max().dropna()
Q_annual_max = Q_annual_max[Q_annual_max > 0]

# Ajuste GEV
shape, loc, scale = genextreme.fit(Q_annual_max, loc=Q_annual_max.mean())

# Períodos de retorno
T = np.array([2, 5, 10, 25, 50, 100, 200, 500])
p_exceed = 1 / T
Q_T = genextreme.ppf(1 - p_exceed, shape, loc, scale)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Série de máximos anuais
axes[0].bar(Q_annual_max.index.year, Q_annual_max.values, color='steelblue', alpha=0.7)
axes[0].set_xlabel('Ano')
axes[0].set_ylabel('Q máximo anual (m³/s)')
axes[0].set_title('Máximos Anuais de Vazão')
axes[0].axhline(Q_annual_max.mean(), color='k', ls='--', lw=1, label=f'Média = {Q_annual_max.mean():.0f}')
axes[0].legend()

# Curva de frequência GEV
T_plot = np.logspace(0, 3, 200)
Q_plot = genextreme.ppf(1 - 1/T_plot, shape, loc, scale)
axes[1].semilogx(T_plot, Q_plot, color='steelblue', lw=2, label='GEV ajustada')

# Posições de plotting empíricas (Gringorten)
n = len(Q_annual_max)
ranks = Q_annual_max.rank()
T_emp = (n + 0.12) / (ranks - 0.44)
axes[1].scatter(T_emp, Q_annual_max.values, color='k', s=20, zorder=5, label='Observado')

for t_val, q_val in zip(T, Q_T):
    axes[1].annotate(f'T={t_val}a\n{q_val:.0f}', xy=(t_val, q_val),
                     fontsize=7, ha='left', color='crimson')

axes[1].set_xlabel('Período de Retorno (anos)')
axes[1].set_ylabel('Vazão (m³/s)')
axes[1].set_title('Análise de Frequência — GEV')
axes[1].legend()

fig.tight_layout()
fig.savefig(FIGDIR / '08_frequencia_cheias.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nEstimativa GEV de período de retorno:')
for t_val, q_val in zip(T, Q_T):
    print(f'  T={t_val:4d} anos → Q = {q_val:7.0f} m³/s')

## 11. Decisões de Pré-processamento — Resumo

Com base na EDA acima, documentamos aqui as decisões para `preprocessor.py`:

In [ ]:
decisions = {
    'log_transform': {
        'decisão': True,
        'justificativa': 'Skewness linear > 1; skewness log próxima de 0 — confirma distribuição log-normal.',
    },
    'max_gap_interpolate': {
        'decisão': 3,
        'justificativa': 'Gaps curtos (1-3 dias) provavelmente são falhas de sensor, não eventos hidrológicos. '
                         'Interpolação linear é conservadora para esse horizonte.',
    },
    'train_split': {
        'decisão': 'Treino: 1940–2000 | Val: 2001–2010 | Teste: 2011–2024',
        'justificativa': 'Teste inclui eventos 2011 e 2023. '
                         'Val inclui período de dados razoavelmente completo antes dos eventos mais recentes.',
    },
    'normalization': {
        'decisão': 'Z-score calculado APENAS no período de treino',
        'justificativa': 'Data leakage: usar estatísticas do conjunto completo vaza informação de val/teste.',
    },
    'consistency_priority': {
        'decisão': 'Nível 2 (consolidado) preferido sobre Nível 1 (bruto)',
        'justificativa': 'Dados consolidados passaram por revisão manual. '
                         'Para datas recentes sem consolidação, aceitar Nível 1.',
    },
}

for key, val in decisions.items():
    print(f'\n── {key} ──')
    print(f'  Decisão:       {val["decisão"]}')
    print(f'  Justificativa: {val["justificativa"]}')

## 12. Próximos Passos

Com a EDA concluída, o fluxo natural é:

1. **`notebooks/02_preprocessing/`** — rodar `preprocessor.py` e validar
   visualmente a série processada vs. bruta
2. **Dados de precipitação** — repetir EDA para as estações pluviométricas
   da bacia (necessárias como features do modelo)
3. **Atributos estáticos da bacia** — extrair área de drenagem, declividade,
   cobertura do solo via MERIT Hydro / MODIS
4. **Formato para o OpenHydroNet** — converter para NetCDF com estrutura
   esperada pelo framework (uma pasta por bacia, arquivos `timeseries.nc`
   e `attributes.csv`)